<a href="https://colab.research.google.com/github/mayankbajaj17/OCM-Divergence/blob/main/Survival_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**STEP 1**

In [ ]:
# Install the necessary library
!pip install lifelines

import pandas as pd
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt
import os
import numpy as np
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# Define paths based on your request
BASE_DIR = '/content/drive/MyDrive/Survival Analysis Gliomas'
os.makedirs(BASE_DIR, exist_ok=True)

# Define input file names and folder
GENE_LIST_FILE = os.path.join(BASE_DIR, "gene_list_signature.xlsx")
# --- NEW: Folder containing individual gene data files ---
DATA_FOLDER_INDIVIDUAL = os.path.join(BASE_DIR, 'GlioVis_Survival_Data_TCGA_Driver')

# Define output folder structure
OUTPUT_FOLDER_INDIVIDUAL = os.path.join(BASE_DIR, 'Per Gene Survival Analysis')
OUTPUT_FOLDER_COMBINED = os.path.join(BASE_DIR, 'Gene Signature Survival Analysis')
os.makedirs(OUTPUT_FOLDER_INDIVIDUAL, exist_ok=True)
os.makedirs(OUTPUT_FOLDER_COMBINED, exist_ok=True)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 10.5 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=2fb5d5d37411f650c0a4eda302c19c582ea90afb51cfc66cd3161967eaec1764
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma
Mounted at /content/drive


**STEP 2**

In [ ]:
# 2. Load Gene List
try:
    gene_list_df = pd.read_excel(GENE_LIST_FILE)
    # Assuming 'Gene_Name' is the column with the official gene symbols (e.g., AHCY)
    GENE_NAMES = gene_list_df['Gene_Name'].tolist()
    print(f"Loaded {len(GENE_NAMES)} genes for analysis.")
except Exception as e:
    print(f"Error loading gene list: {e}")
    GENE_NAMES = [] # Safety measure

# 3. Aggregation of Individual Gene Data Files
master_data = None
gene_expression_columns = []

print("\n--- Starting Data Aggregation ---")

for i, gene in enumerate(GENE_NAMES):
    file_path = os.path.join(DATA_FOLDER_INDIVIDUAL, f'{gene}.csv')

    if os.path.exists(file_path):
        try:
            # Read the gene's data file
            gene_df = pd.read_csv(file_path)

            # Standardize column names based on your uploaded example
            gene_df = gene_df.rename(columns={'survival': 'Time', 'status': 'Event', 'mRNA': gene})

            # Select essential clinical columns from the first file loaded
            if master_data is None:
                # Initialize master_data with shared clinical columns
                master_data = gene_df[['Sample', 'Time', 'Event', 'IDH_status']].copy().set_index('Sample')
                print(f"Initialized master data with {len(master_data)} samples.")

            # Merge the current gene's expression (renamed to the gene symbol)
            expression_data = gene_df[['Sample', gene]].set_index('Sample')
            master_data = master_data.merge(expression_data, left_index=True, right_index=True, how='inner')
            gene_expression_columns.append(gene)

            if (i + 1) % 10 == 0:
                print(f"Processed {i + 1} files...")

        except Exception as e:
            print(f"Skipping gene {gene} due to error during file loading/merging: {e}")
    else:
        print(f"File not found for gene: {gene}")

# Final Master Data Processing
full_data = master_data.reset_index()

# 4. Calculate Combined Gene Set Score
if gene_expression_columns:
    full_data['Combined_Score'] = full_data[gene_expression_columns].mean(axis=1)
    print(f"\nSuccessfully aggregated data for {len(gene_expression_columns)} genes.")
    print(f"Final combined dataset size: {len(full_data)} samples.")
else:
    print("\nFatal Error: No gene data files were successfully loaded. Stopping.")
    # Exit or raise error if necessary

Loaded 64 genes for analysis.

--- Starting Data Aggregation ---
File not found for gene: AGXT2
Initialized master data with 667 samples.
File not found for gene: AHCYL2
File not found for gene: ALDH1L1
File not found for gene: AMT
File not found for gene: ATIC
File not found for gene: BAAT
File not found for gene: BHMT
File not found for gene: BHMT2
File not found for gene: CBS
File not found for gene: CDO1
File not found for gene: CEPT1
File not found for gene: CHPT1
File not found for gene: CSAD
File not found for gene: CTH
File not found for gene: DHFR
File not found for gene: DMGDH
File not found for gene: DNMT1
File not found for gene: DNMT3A
File not found for gene: ETNK1
File not found for gene: ETNK2
File not found for gene: FTCD
File not found for gene: GAD1
File not found for gene: GAD2
File not found for gene: GART
File not found for gene: GLDC
File not found for gene: GLRX
File not found for gene: GNMT
File not found for gene: GPX4
Processed 40 files...
File not found for 

**STEP 3**

In [ ]:
import statsmodels.stats.multitest as multitest
from lifelines import CoxPHFitter

# --- Step 1A: Get Cox P-values for all genes (All IDH cohort) ---
cox_p_values = {}
# Use the full cohort for initial filtering (All IDH)
df_cohort = full_data[['Time', 'Event'] + gene_expression_columns].dropna()

for gene in gene_expression_columns:
    try:
        cph = CoxPHFitter()
        # The Cox model uses the single gene as the only predictor
        cox_data = df_cohort[['Time', 'Event', gene]].copy()

        # Fit the model
        cph.fit(cox_data, duration_col='Time', event_col='Event', formula=gene)

        # Store the p-value for the gene
        cox_p_values[gene] = cph.summary.loc[gene, 'p']
    except Exception as e:
        print(f"Skipping Cox model for gene {gene} due to error: {e}")
        cox_p_values[gene] = 1.0 # Assign high p-value if model fails

# --- Step 1B: Apply FDR Correction (Benjamini-Hochberg) ---
gene_p_df = pd.Series(cox_p_values).to_frame(name='raw_p_value')
gene_p_df.reset_index(names='Gene', inplace=True)

# Apply Benjamini-Hochberg FDR correction
# The q-value (FDR) is the adjusted p-value
reject, p_adjusted, _, _ = multitest.multipletests(
    gene_p_df['raw_p_value'],
    alpha=0.1, # Set FDR threshold to 10% (a common threshold in discovery)
    method='fdr_bh'
)

gene_p_df['FDR_q_value'] = p_adjusted
gene_p_df['Is_Significant'] = reject

# --- Step 1C: Filter the Gene List ---
FDR_THRESHOLD = 0.1
significant_genes = gene_p_df[gene_p_df['FDR_q_value'] <= FDR_THRESHOLD]

# Store the final list of genes to be used in the combined score
FINAL_GENE_SET = significant_genes['Gene'].tolist()

print(f"\n--- FDR Filtering Results ---")
print(f"Total genes tested: {len(gene_expression_columns)}")
print(f"Genes filtered (FDR < {FDR_THRESHOLD}): {len(FINAL_GENE_SET)}")
print(f"Genes dropped (non-prognostic): {len(gene_expression_columns) - len(FINAL_GENE_SET)}")


--- FDR Filtering Results ---
Total genes tested: 20
Genes filtered (FDR < 0.1): 16
Genes dropped (non-prognostic): 4


**STEP 4**

In [ ]:
from scipy.stats import zscore

if FINAL_GENE_SET:
    # --- Step 2A: Z-Score Normalization on Filtered Genes ---

    # Select only the significant genes from the full data
    expression_data_filtered = full_data[FINAL_GENE_SET].copy()

    # Apply the Z-score transformation
    zscored_expression_data_filtered = expression_data_filtered.apply(zscore, axis=0)

    # Handle cases where filtered genes might still be constant (highly unlikely now)
    zscored_expression_data_filtered.fillna(0, inplace=True)
    zscored_expression_data_filtered.replace([np.inf, -np.inf], 0, inplace=True)

    # --- Step 2B: Recalculate Combined Score using Z-Scores of Filtered Genes ---

    # Calculate the average of the Z-scores across all filtered genes for each patient
    full_data['Combined_Score_Z_Filtered'] = zscored_expression_data_filtered.mean(axis=1)

    print("\nSuccessfully calculated 'Combined_Score_Z_Filtered'.")
else:
    print("\nFatal Error: No genes passed the FDR filter. Cannot calculate combined score.")


Successfully calculated 'Combined_Score_Z_Filtered'.


**STEP 5**

In [ ]:
from lifelines.statistics import logrank_test, multivariate_logrank_test
# Removed: survival_difference_at_fixed_point_in_time_test

def plot_km_and_stats(data_df, analysis_column, analysis_name, condition_title, output_folder):
    """Generates the KM plot, calculates stats (using multivariate_logrank_test for Wilcoxon), and saves the figure."""

    # Map condition titles to desired folder names
    condition_folder_map = {
        'Pan-Glioma Cohort': 'Pan-Glioma_Cohort',
        'IDH1 Mutant Cohort': 'IDH1_Mutant_Cohort',
        'IDH1 Wildtype Cohort': 'IDH1_Wildtype_Cohort'
    }
    mapped_condition_name = condition_folder_map.get(condition_title, condition_title.replace(" ", "_"))

    # 1. Grouping (High vs. Low) based on the median expression/score
    cutoff = data_df[analysis_column].median()
    data_df['Group'] = np.where(data_df[analysis_column] > cutoff, 'High', 'Low')

    T = data_df['Time']
    E = data_df['Event']
    groups = data_df['Group']

    # Filter cohorts
    data_high = data_df[groups == 'High']
    data_low = data_df[groups == 'Low']

    if len(data_high) < 5 or len(data_low) < 5:
        print(f"Skipping {analysis_name} ({condition_title}): Not enough samples in one group.")
        return

    # 2. KM Plotting and Median Survival Calculation

    # Initialize separate KM objects for the groups
    kmf_high = KaplanMeierFitter()
    kmf_low = KaplanMeierFitter()

    fig, ax = plt.subplots(figsize=(8, 6))

    # Fit and plot Low group
    kmf_low.fit(data_low['Time'], data_low['Event'], label=f'{analysis_name} Low (n={len(data_low)})')
    kmf_low.plot(ax=ax, ci_show=True, c='C1') # Orange for Low
    median_low = kmf_low.median_survival_time_

    # Fit and plot High group
    kmf_high.fit(data_high['Time'], data_high['Event'], label=f'{analysis_name} High (n={len(data_high)})')
    kmf_high.plot(ax=ax, ci_show=True, c='C0') # Blue for High
    median_high = kmf_high.median_survival_time_

    # 3. Statistical Tests

    # Log-rank Test (default weighting)
    results_lr = logrank_test(data_high['Time'], data_low['Time'], data_high['Event'], data_low['Event'])
    log_rank_p = results_lr.p_value

    # Wilcoxon (Peto-Peto) Test - Use multivariate_logrank_test with Peto's weighting
    # This correctly performs the Wilcoxon (Peto-Peto) test by setting the weightings argument.

    # Combine data for multivariate test
    T_all = data_df['Time']
    E_all = data_df['Event']
    groups_all = data_df['Group']

    results_wilcoxon = multivariate_logrank_test(
        T_all, groups_all, E_all, weightings='wilcoxon'
    )
    wilcoxon_p = results_wilcoxon.p_value

    # 4. Cox Proportional Hazard Model (for HR, CI, and Cox p)
    try:
        cph = CoxPHFitter()
        cox_data = data_df[['Time', 'Event']].copy()
        cox_data['group_encoded'] = np.where(data_df['Group'] == 'High', 1, 0)

        cph.fit(cox_data, duration_col='Time', event_col='Event', formula='group_encoded')

        summary = cph.summary.loc['group_encoded']
        hr = summary['exp(coef)']
        hr_ci_lower = summary['exp(coef) lower 95%']
        hr_ci_upper = summary['exp(coef) upper 95%']
        cox_p = summary['p']

        stats_text = (
            f"Median High = {median_high:.1f}\n"
            f"Median Low = {median_low:.1f}\n"
            f"HR (High vs Low) = {hr:.2f}\n"
            f"95% CI = [{hr_ci_lower:.2f} - {hr_ci_upper:.2f}]\n"
            f"Log-rank p = {log_rank_p:.3g}\n"
            f"Wilcoxon p = {wilcoxon_p:.3g}\n"
            f"Cox p = {cox_p:.3g}"
        )
    except Exception as e:
        stats_text = (
            f"Median High = {median_high:.1f}\n"
            f"Median Low = {median_low:.1f}\n"
            f"HR = NA\n95% CI = NA\n"
            f"Log-rank p = {log_rank_p:.3g}\n"
            f"Wilcoxon p = {wilcoxon_p:.3g}\n"
            f"Cox p = NA"
        )

    # 5. Add Stats and Aesthetics
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle="round,pad=0.5", fc="white", alpha=0.8))

    ax.set_title(f'{analysis_name} ({condition_title})')
    ax.set_xlabel('Time (Months)')
    ax.set_ylabel('Survival probability')
    ax.set_ylim(0, 1.05)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='lower left')

    # Save the figure
    file_name = f'{analysis_name.replace(" ", "_")}_{mapped_condition_name}_KM.png'
    output_path_final = os.path.join(output_folder, mapped_condition_name)
    os.makedirs(output_path_final, exist_ok=True)

    # Add a check to ensure the directory exists before saving
    if os.path.exists(output_path_final):
        plt.savefig(os.path.join(output_path_final, file_name))
        plt.close(fig)
        print(f"Saved: {file_name} to {output_path_final}")
    else:
        print(f"Warning: Could not create directory {output_path_final}. Skipping plot save for {file_name}.")
        plt.close(fig)


**STEP 6**

In [ ]:
# Check if aggregation was successful before proceeding
if 'Combined_Score' not in full_data.columns or full_data.empty:
    print("Cannot proceed with analysis due to data aggregation failure.")
else:
    # Define the conditions for filtering the data
    CONDITIONS = [
        ('Pan-Glioma Cohort', full_data),
        ('IDH1 Mutant Cohort', full_data[full_data['IDH_status'] == 'Mutant']),
        ('IDH1 Wildtype Cohort', full_data[full_data['IDH_status'] == 'Wild_type']),
    ]

    # =======================================================
    # Run Analysis for Individual Genes (a, b, c)
    # =======================================================
    print("\n--- Starting Individual Gene Analysis (a, b, c) ---")

    for gene in gene_expression_columns:
        for condition_title, condition_data in CONDITIONS:
            # Filter and create a copy for the current subset
            data_subset = condition_data[['Time', 'Event', 'IDH_status', gene]].dropna().copy()

            if not data_subset.empty:
                plot_km_and_stats(
                    data_df=data_subset,
                    analysis_column=gene,
                    analysis_name=gene,
                    condition_title=condition_title,
                    output_folder=OUTPUT_FOLDER_INDIVIDUAL
                )

# Assuming the 'plot_km_and_stats' function definition is loaded.

# Define the conditions for filtering the data
CONDITIONS = [
    ('Pan-Glioma Cohort', full_data),
    ('IDH1 Mutant Cohort', full_data[full_data['IDH_status'] == 'Mutant']),
    ('IDH1 Wildtype Cohort', full_data[full_data['IDH_status'] == 'Wild_type']),
]

# =======================================================
# Run Analysis for Combined Gene Set (d, e, f) using Z-Scores (Filtered)
# =======================================================
print("\n--- Starting Final Combined Gene Set Analysis (d, e, f) ---")

if 'Combined_Score_Z_Filtered' in full_data.columns:
    for condition_title, condition_data in CONDITIONS:
        # Filter and create a copy for the current subset
        data_subset = condition_data[['Time', 'Event', 'IDH_status', 'Combined_Score_Z_Filtered']].dropna().copy()

        if not data_subset.empty:
            plot_km_and_stats(
                data_df=data_subset,
                analysis_column='Combined_Score_Z_Filtered',
                analysis_name='Gene_Set_Final_Score',
                condition_title=condition_title,
                output_folder=OUTPUT_FOLDER_COMBINED
            )
else:
    print("Cannot perform combined analysis: 'Combined_Score_Z_Filtered' column is missing.")

print("\n--- Final Robust Combined Score Analyses Complete ---")


--- Starting Individual Gene Analysis (a, b, c) ---
Saved: AHCY_Pan-Glioma_Cohort_KM.png to /content/drive/MyDrive/Survival Analysis Gliomas/Per Gene Survival Analysis/Pan-Glioma_Cohort
Saved: AHCY_IDH1_Mutant_Cohort_KM.png to /content/drive/MyDrive/Survival Analysis Gliomas/Per Gene Survival Analysis/IDH1_Mutant_Cohort
Saved: AHCY_IDH1_Wildtype_Cohort_KM.png to /content/drive/MyDrive/Survival Analysis Gliomas/Per Gene Survival Analysis/IDH1_Wildtype_Cohort
Saved: BCAT1_Pan-Glioma_Cohort_KM.png to /content/drive/MyDrive/Survival Analysis Gliomas/Per Gene Survival Analysis/Pan-Glioma_Cohort
Saved: BCAT1_IDH1_Mutant_Cohort_KM.png to /content/drive/MyDrive/Survival Analysis Gliomas/Per Gene Survival Analysis/IDH1_Mutant_Cohort
Saved: BCAT1_IDH1_Wildtype_Cohort_KM.png to /content/drive/MyDrive/Survival Analysis Gliomas/Per Gene Survival Analysis/IDH1_Wildtype_Cohort
Saved: BCAT2_Pan-Glioma_Cohort_KM.png to /content/drive/MyDrive/Survival Analysis Gliomas/Per Gene Survival Analysis/Pan-Gli

**STEP 7**

In [ ]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter
import os

# --- Define the output path for the CSV file ---
# You can change this path if you want it saved elsewhere.
# It is recommended to save it in your combined results folder.
OUTPUT_CSV_PATH = "/content/drive/MyDrive/Survival Analysis Gliomas/Bivariate_Cox_Summary.csv"

# Ensure the output directory exists
os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)

# --- Prep Data (Assuming full_data DataFrame and FINAL_GENE_SET are defined) ---
# NOTE: This assumes 'full_data' and 'Combined_Score_Z_Filtered' already exist from prior steps.
df_cox = full_data[['Time', 'Event', 'IDH_status', 'Combined_Score_Z_Filtered']].dropna().copy()

# Encode IDH Status (Wildtype=1, Mutant=0)
df_cox['IDH_encoded'] = np.where(df_cox['IDH_status'] == 'Wild_type', 1, 0)

# --- Step 1: Run the Bivariate Cox Model ---
print("\n--- Starting Bivariate Cox Analysis (IDH Status + Gene Score) ---")

try:
    cph_bivariate = CoxPHFitter()

    # The formula includes both predictors: Gene Score and IDH Status
    cph_bivariate.fit(
        df_cox,
        duration_col='Time',
        event_col='Event',
        formula='Combined_Score_Z_Filtered + IDH_encoded'
    )

    # Print the summary (as before, for immediate review)
    print(cph_bivariate.print_summary())

    # --- Step 2: Save the Summary to CSV ---
    # The .summary attribute is a pandas DataFrame
    summary_df = cph_bivariate.summary

    # Save the DataFrame to the specified path
    summary_df.to_csv(OUTPUT_CSV_PATH, index=True, float_format='%.4g')

    print(f"\nSuccessfully saved Bivariate Cox Summary to: {OUTPUT_CSV_PATH}")

except Exception as e:
    print(f"Bivariate Cox model failed: {e}")
    print("Saving CSV skipped due to error.")


--- Starting Bivariate Cox Analysis (IDH Status + Gene Score) ---


<lifelines.CoxPHFitter: fitted with 660 total observations, 425 right-censored observations>
             duration col = 'Time'
                event col = 'Event'
      baseline estimation = breslow
   number of observations = 660
number of events observed = 235
   partial log-likelihood = -1159.19
         time fit was run = 2025-11-20 07:06:29 UTC

---
                           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                  
Combined_Score_Z_Filtered  0.57      1.78      0.28            0.02            1.13                1.02                3.09
IDH_encoded                2.18      8.82      0.16            1.86            2.49                6.42               12.12

                           cmp to     z      p  -log2(p)
covariate                                               
Combined_Score_Z_Filtered    0.00  2.03   0.04      4.57
IDH_encoded                  0.00 13.44 <0.005    134.44
---
Concordance = 0.81
Partial AIC = 2322.38
log-likelihood ratio test = 268.45 on 2 df
-log2(p) of ll-ratio test = 193.64

None

Successfully saved Bivariate Cox Summary to: /content/drive/MyDrive/Survival Analysis Gliomas/Combined_Gene_KM_Plots/Bivariate_Cox_Summary.csv
